In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

from minigrad import MLP
from augment import augment_batch
from data import load_mnist, one_hot, build_or_load_augmented_pool
from train import train, evaluate, predict

In [ ]:
x_train, y_train, x_test, y_test = load_mnist('data')
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

In [ ]:
def show_images(images, title_texts):
    cols = 5
    rows = int(len(images) / cols) + 1
    plt.figure(figsize=(30, 20))
    for index, (image, title_text) in enumerate(zip(images, title_texts), start=1):
        plt.subplot(rows, cols, index)
        plt.imshow(image.reshape(28, 28), cmap=plt.cm.gray)
        if title_text:
            plt.title(title_text, fontsize=15)

images_2_show, titles_2_show = [], []
for _ in range(10):
    r = random.randint(0, len(x_train) - 1)
    images_2_show.append(x_train[r])
    titles_2_show.append(f'training image [{r}] = {y_train[r]}')
for _ in range(5):
    r = random.randint(0, len(x_test) - 1)
    images_2_show.append(x_test[r])
    titles_2_show.append(f'test image [{r}] = {y_test[r]}')
show_images(images_2_show, titles_2_show)

In [ ]:
y_train_enc = one_hot(y_train, 10)

In [ ]:
# demo augmentation, make sure it works
n = 8
orig = x_train[:n]
aug = augment_batch(orig)

fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
for i in range(n):
    axes[0, i].imshow(orig[i].reshape(28, 28), cmap='gray_r', vmin=0, vmax=1)
    axes[0, i].set_title(f'orig {y_train[i]}', fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(aug[i].reshape(28, 28), cmap='gray_r', vmin=0, vmax=1)
    axes[1, i].set_title('aug', fontsize=9)
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
x_pool, y_pool = build_or_load_augmented_pool(
    x_train, y_train_enc, 'data/mnist_augmented_pool.npz', K=4,
)

In [ ]:
m = MLP(784, [100, 10], dropout_p=0.2)

In [ ]:
loss_vals = train(
    m, x_pool, y_pool,
    x_test=x_test, y_test=y_test,
    num_epochs=500, batch_size=128, lr=1.0,
)

In [ ]:
# np.savez('models/higherLR.npz', *[p.data for p in m.parameters()])

In [ ]:
# EXPORT TO BIN for frontend
# d = np.load('models/higherLR.npz')
# np.concatenate([
#     d['arr_0'].astype(np.float32).ravel(),  # W1
#     d['arr_1'].astype(np.float32).ravel(),  # b1
#     d['arr_2'].astype(np.float32).ravel(),  # W2
#     d['arr_3'].astype(np.float32).ravel(),  # b2
# ]).tofile('../../web/weights.bin')

In [ ]:
# accuracy on training set
evaluate(m, x_pool, np.argmax(y_pool, axis=1))

In [ ]:
# show some misclassified test digits
preds = predict(m, x_test)

wrong = np.where(preds != y_test)[0]
print(f'{len(wrong)} / {len(y_test)} misclassified ({100 * len(wrong) / len(y_test):.2f}%)')

n_show = min(25, len(wrong))
sample = np.random.choice(wrong, size=n_show, replace=False)
show_images(
    [x_test[i] for i in sample],
    [f'pred {preds[i]} / true {y_test[i]}' for i in sample],
)

In [ ]:
# plot loss values for each batch
plt.plot(loss_vals)
plt.xlim(left=0)
plt.ylim(bottom=0)
plt.show()